# Лабораторная работа 6
# Выравнивание списка, состоящего из итерируемых объектов, на основе рекурсии

**Рекурсивная функция** -- это функция, которая вызывает сама себя, и при каждом
очередном вызове может использовать объекты, созданные во время предыдущего
вызова.

**Стек вызовов** -- это область памяти, в которой выполняются функции. При каждом
вызове функции создается **фрейм** -- фрагмент памяти, в котором содержится:
- информация о текущем выполнении функции;
- объекты, полученные функцией для обработки;
- локальные пременные;
- сведения о строке программы, к которой нужно вернуться после выполнения
функции.
Фреймы добаляются и удаляются по принципу стека

Для того, чтобы рекурсивная функция реализовывала конечное число вызовов,
необходимо в рекурсивной функции обрабатывать ситуацию, когда функция
возвращает результат. Такая "граничная" ситуация называется **базой рекурсии**
.


In [1]:
import sys
sys.getrecursionlimit()

3000

In [2]:
def sum(n):
 """суммирует числа от 0 до n>=0"""
 match n:
     case 0: return n # база рекурсии
     case _: return n + sum(n-1)    
sum(0), sum(2), sum(10)

(0, 3, 55)

In [3]:
sum = lambda n: n + sum(n-1) if n else n
sum(0), sum(2), sum(10)

(0, 3, 55)

## Задание 6.1. Выравнивание вложенных списков


### Задание 6.1.1. Рекурсивная функция flatten_list_v1

In [4]:
it_list = [[1],[2,[3]],[[[[[4]]]]],5,6,7]

In [5]:
?isinstance

Signature: isinstance(obj, class_or_tuple, /)
Docstring:
Return whether an object is an instance of a class or of a subclass thereof.

A tuple, as in ``isinstance(x, (A, B, ...))``, may be given as the target to
check against. This is equivalent to ``isinstance(x, A) or isinstance(x, B)
or ...`` etc.
Type:      builtin_function_or_method

In [6]:
def flatten_list_v1(nested_obj):
 result = []
 if isinstance(nested_obj,list):
     for item in nested_obj:
         result += flatten_list_v1(item) # рекурсия
 else:
     result += [nested_obj]
 return result

In [7]:
flatten_list_v1(it_list)

[1, 2, 3, 4, 5, 6, 7]

### Задание 6.1.2. Генераторная функция на основе рекурсии flatten_list_v2


In [10]:
def flatten_list_v2(nested_obj):
    if isinstance(nested_obj, list):
        for item in nested_obj:
            # Рекурсивно получаем элементы из вложенного списка
            for element in flatten_list_v2(item):
                yield element  # выдаём каждый элемент наружу
    else:
        yield nested_obj  # не список — просто выдаём сам объект

In [11]:
[x for x in flatten_list_v2(it_list)]

[1, 2, 3, 4, 5, 6, 7]

In [12]:
def flatten_list_v2_2(nested_obj):
    if isinstance(nested_obj, list):
        for item in nested_obj:
            yield from flatten_list_v2_2(item)  # делегируем всё вложенному генератору
    else:
        yield nested_obj

In [13]:
[x for x in flatten_list_v2_2(it_list)]

[1, 2, 3, 4, 5, 6, 7]

### Задание 6.1.3. Функция flatten_list: создание, документирование, тестирование

In [20]:
def flatten_list(nested_obj, gen=False):
    """
    Преобразует вложенный список в плоский список или генератор.

    Parameters
    ----------
    nested_obj : list or any
        Вложенный список, который необходимо преобразовать.
        Может содержать элементы любых типов и произвольную глубину вложенности.
    gen : bool, optional
        Определяет тип возвращаемого значения:
        - False (по умолчанию): возвращает плоский список
        - True: возвращает генератор, последовательно выдающий элементы

    Returns
    -------
    list or generator
        Если gen=False: возвращает плоский список всех элементов nested_obj.
        Если gen=True: возвращает генератор, выдающий элементы по одному.
    """
    
    if gen:
        # Генераторная версия (как flatten_list_v2)
        if isinstance(nested_obj, list):
            for item in nested_obj:
                yield from flatten_list(item, gen=True)
        else:
            yield nested_obj
    else:
        # Версия, возвращающая список (как flatten_list_v1)
        result = []
        if isinstance(nested_obj, list):
            for item in nested_obj:
                result.extend(flatten_list(item, gen=False))
        else:
            result.append(nested_obj)
        return result

In [15]:
nested2 = [1, [2, 3], [4, [5, 6], 7]]
flat_gen = flatten_list(nested2, gen=True)
for item in flat_gen:
    print(item, end=' ')

1 2 3 4 5 6 7 

In [16]:
nested3 = [[1, 2], [3, [4, 5]], 6]
flat_gen = flatten_list(nested3, gen=True)
print(type(flat_gen))  
print(list(flat_gen))

<class 'generator'>
[1, 2, 3, 4, 5, 6]


In [21]:
nested5 = [1, [2, [3, [4, 5]]]]
result = flatten_list(nested5)
print(result)  
print(type(result))  

<generator object flatten_list at 0x000001961D0B7A40>
<class 'generator'>


## Задание 6.2. Выравнивание вложенных итерируемых объектов произвольного типа

In [30]:
it_list

[[1], [2, [3]], [[[[[4]]]]], 5, 6, 7]

In [31]:
it_tuple = (((7),(8)),(9),10)

In [32]:
it_str = "abcdefgh"

In [33]:
it_dict = {"key1": 11, "key2": {"key3": 12}}

In [34]:
it_gen = ([i,i**2,i**3] for i in range(5))

In [36]:
it = [it_list, it_tuple, it_str, it_dict, it_gen]

In [37]:
['__iter__' in dir(x) for x in it]

[True, True, True, True, True]

In [38]:
[hasattr(x, '__iter__') for x in it]

[True, True, True, True, True]

In [39]:
?hasattr

Signature: hasattr(obj, name, /)
Docstring:
Return whether the object has an attribute with the given name.

This is done by calling getattr(obj, name) and catching AttributeError.
Type:      builtin_function_or_method

### Задание 6.2.1. Рекурсивная функция flatten_it_v1

In [40]:
def flatten_it_v1(nested_obj):
    result = []
    # Проверяем, является ли объект итерируемым (имеет метод __iter__)
    if hasattr(nested_obj, '__iter__'):
        # Дополнительное условие: если это строка, не итерируем её,
        # а обрабатываем как обычный объект (добавляем целиком)
        if isinstance(nested_obj, str):
            result.append(nested_obj)
        # Если это словарь, нужно итерировать и по ключам, и по значениям
        elif isinstance(nested_obj, dict):
            for key, value in nested_obj.items():
                # Рекурсивно обрабатываем ключ и значение
                result.extend(flatten_it_v1(key))
                result.extend(flatten_it_v1(value))
        else:
            # Для всех остальных итерируемых объектов (списки, кортежи, множества и т.д.)
            for item in nested_obj:
                # Рекурсивно выравниваем каждый элемент
                result.extend(flatten_it_v1(item))
    else:
        # Если объект не итерируемый, добавляем его как есть
        result.append(nested_obj)
    return result

In [41]:
print(flatten_it_v1(it))

[1, 2, 3, 4, 5, 6, 7, 7, 8, 9, 10, 'abcdefgh', 'key1', 11, 'key2', 'key3', 12, 0, 0, 0, 1, 1, 1, 2, 4, 8, 3, 9, 27, 4, 16, 64]


### Задание 6.2.2. Генераторная функция на основе рекурсии flatten_it_v2

In [42]:
def flatten_it_v2(nested_obj):
    # Проверяем, итерируемый ли объект (кроме строк)
    if hasattr(nested_obj, '__iter__') and not isinstance(nested_obj, str):
        # Если это словарь — обрабатываем ключи и значения
        if isinstance(nested_obj, dict):
            for key, value in nested_obj.items():
                # Рекурсивно отдаём ключ и значение
                yield from flatten_it_v2(key)
                yield from flatten_it_v2(value)
        else:
            # Для всех остальных итерируемых объектов
            for item in nested_obj:
                # Рекурсивно делегируем генератору вложенного объекта
                yield from flatten_it_v2(item)
    else:
        # Базовый случай: неитерируемый или строка — выдаём сам объект
        yield nested_obj

In [47]:
print(list(flatten_it_v2(it)))

[1, 2, 3, 4, 5, 6, 7, 7, 8, 9, 10, 'abcdefgh', 'key1', 11, 'key2', 'key3', 12]


In [48]:
# Заново определяем все тестовые данные
it_list = [[1], [2, [3]], [[[[[4]]]], 5, 6, 7]]
it_tuple = (((7),(8)),(9),10)
it_str = "abcdefgh"
it_dict = {"key1": 11, "key2": {"key3": 12}}

it_gen_v1 = ([i, i**2, i**3] for i in range(5))
it_v1 = [it_list, it_tuple, it_str, it_dict, it_gen_v1]

it_gen_v2 = ([i, i**2, i**3] for i in range(5))
it_v2 = [it_list, it_tuple, it_str, it_dict, it_gen_v2]

# Теперь тестируем
result_v1 = flatten_it_v1(it_v1)
result_v2 = list(flatten_it_v2(it_v2))

print(f"Результаты совпадают: {result_v1 == result_v2}")
print(f"Длина результата: {len(result_v1)}")
print(f"result_v2: {result_v2}")

Результаты совпадают: True
Длина результата: 32
result_v2: [1, 2, 3, 4, 5, 6, 7, 7, 8, 9, 10, 'abcdefgh', 'key1', 11, 'key2', 'key3', 12, 0, 0, 0, 1, 1, 1, 2, 4, 8, 3, 9, 27, 4, 16, 64]


### Задание 6.2.3. Функция flatten_it: создание, документирование, тестирование

In [49]:
def flatten_it(nested_obj, gen=False):
    """
    Выравнивает вложенную структуру из различных итерируемых объектов.
    
    Parameters
    ----------
    nested_obj : any
        Объект для выравнивания. Может быть любым итерируемым объектом
        (список, кортеж, словарь, генератор, файловый объект и т.д.)
        или неитерируемым объектом.
    gen : bool, default=False
        Определяет тип возвращаемого значения:
        - False (по умолчанию): возвращает плоский список
        - True: возвращает генератор, выдающий элементы по одному
    
    Returns
    -------
    list or generator
        Если gen=False: возвращает список всех элементов без сохранения структуры
        Если gen=True: возвращает генератор, последовательно выдающий элементы
    """
    
    def _flatten_generator(nested_obj):
        # Проверяем, итерируемый ли объект (и не строка)
        if hasattr(nested_obj, '__iter__') and not isinstance(nested_obj, str):
            # Особая обработка для словарей
            if isinstance(nested_obj, dict):
                for key, value in nested_obj.items():
                    yield from _flatten_generator(key)
                    yield from _flatten_generator(value)
            else:
                # Для всех остальных итерируемых объектов
                for item in nested_obj:
                    yield from _flatten_generator(item)
        else:
            # Базовый случай: неитерируемый объект или строка
            yield nested_obj
    
    # Создаём генератор
    generator = _flatten_generator(nested_obj)
    
    # Возвращаем либо список, либо генератор в зависимости от параметра gen
    if gen:
        return generator
    else:
        return list(generator)

In [50]:
data1 = [1, [2, 3], {'a': 4, 'b': [5, 6]}, "hello", (7, 8)]
result1 = flatten_it(data1)
print(f"Входные данные: {data1}")
print(f"Результат: {result1}")
print(f"Тип результата: {type(result1)}\n")

Входные данные: [1, [2, 3], {'a': 4, 'b': [5, 6]}, 'hello', (7, 8)]
Результат: [1, 2, 3, 'a', 4, 'b', 5, 6, 'hello', 7, 8]
Тип результата: <class 'list'>



In [51]:
data2 = [[1, 2], [3, [4, 5]], {"x": 6, "y": 7}]
result2 = flatten_it(data2, gen=True)
print(f"Входные данные: {data2}")
print(f"Результат: {result2}")
print(f"Тип результата: {type(result2)}")
print(f"Элементы из генератора: {list(result2)}\n")

Входные данные: [[1, 2], [3, [4, 5]], {'x': 6, 'y': 7}]
Результат: <generator object flatten_it.<locals>._flatten_generator at 0x000001961A3897B0>
Тип результата: <class 'generator'>
Элементы из генератора: [1, 2, 3, 4, 5, 'x', 6, 'y', 7]



In [52]:
data3 = [1, [2, [3, [4, [5]]]], (6, 7, (8, 9)), range(10, 13)]
result3 = flatten_it(data3)
print(f"Входные данные: {data3}")
print(f"Результат: {result3}")

Входные данные: [1, [2, [3, [4, [5]]]], (6, 7, (8, 9)), range(10, 13)]
Результат: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


## Задание 6.3. Обработка циклических объектов

In [22]:
it_list_cyclic = it_list[:]
it_list_cyclic.append(it_list_cyclic)

In [23]:
it_list_cyclic

[[1], [2, [3]], [[[[[4]]]]], 5, 6, 7, [...]]

In [24]:
def flatten_list_v1_1(nested_obj):
 result = []
 if isinstance(nested_obj,list):
     for item in nested_obj:
        if item is nested_obj:
            # генерация исключения 
            raise ValueError("Cyclic list is found")
        else:
            result += flatten_list_v1(item)
 else:
     result += [nested_obj]
 return result          

In [25]:
flatten_list_v1_1(it_list)

[1, 2, 3, 4, 5, 6, 7]

In [29]:
try:
    print(flatten_list_v1_1(it_list_cyclic))
except ValueError as e:
    print(e) 

Cyclic list is found


In [53]:
def flatten_it(nested_obj, gen=False):
    """
    Выравнивает вложенную структуру из различных итерируемых объектов.
    
    Parameters
    ----------
    nested_obj : any
        Объект для выравнивания. Может быть любым итерируемым объектом
        (список, кортеж, словарь, генератор, файловый объект и т.д.)
        или неитерируемым объектом.
    gen : bool, default=False
        Определяет тип возвращаемого значения:
        - False (по умолчанию): возвращает плоский список
        - True: возвращает генератор, выдающий элементы по одному
    
    Returns
    -------
    list or generator
        Если gen=False: возвращает список всех элементов без сохранения структуры
        Если gen=True: возвращает генератор, последовательно выдающий элементы
    
    Raises
    ------
    ValueError
        Если обнаружен циклический список или циклический словарь.
        Циклическая структура возникает, когда объект содержит ссылку сам на себя,
        что приводит к бесконечной рекурсии.
    
    Notes
    -----
    Особенности обработки:
    - Строки не итерируются и возвращаются целиком
    - Словари обрабатываются: выдаются как ключи, так и значения
    - Генераторы и файловые объекты корректно выравниваются
    - Функция рекурсивно обходит все вложенные структуры
    - Для обнаружения циклов используется множество id() уже обработанных объектов
    
    """
    
    # Множество для хранения id уже обработанных объектов
    # Используем set для отслеживания циклических ссылок
    processed_ids = set()
    
    def _flatten_generator(nested_obj, path=None):
        """
        Внутренняя генераторная функция для рекурсивного обхода.
        
        Parameters
        ----------
        nested_obj : any
            Объект для обработки
        path : list, optional
            Список id объектов на пути текущей рекурсии (для отладки)
        """
        if path is None:
            path = []
        
        # Получаем id текущего объекта
        obj_id = id(nested_obj)
        
        # Проверяем, не обрабатывали ли мы этот объект ранее
        # (глобальная проверка на циклические ссылки)
        if obj_id in processed_ids:
            # Определяем тип циклического объекта
            if isinstance(nested_obj, list):
                raise ValueError(f"Обнаружен циклический список: объект уже был обработан. Путь: {path}")
            elif isinstance(nested_obj, dict):
                raise ValueError(f"Обнаружен циклический словарь: объект уже был обработан. Путь: {path}")
            else:
                raise ValueError(f"Обнаружена циклическая ссылка на объект типа {type(nested_obj).__name__}")
        
        # Проверяем, итерируемый ли объект (и не строка)
        if hasattr(nested_obj, '__iter__') and not isinstance(nested_obj, str):
            # Особая обработка для словарей
            if isinstance(nested_obj, dict):
                # Добавляем id словаря в обработанные
                processed_ids.add(obj_id)
                try:
                    for key, value in nested_obj.items():
                        # Проверяем ключ и значение на цикличность
                        yield from _flatten_generator(key, path + [f"dict_key_{key}"])
                        yield from _flatten_generator(value, path + [f"dict_value_{key}"])
                finally:
                    # Удаляем id после обработки (важно для независимых веток)
                    processed_ids.remove(obj_id)
            else:
                # Для всех остальных итерируемых объектов
                # Добавляем id в обработанные
                processed_ids.add(obj_id)
                try:
                    for idx, item in enumerate(nested_obj):
                        yield from _flatten_generator(item, path + [f"{type(nested_obj).__name__}[{idx}]"])
                finally:
                    # Удаляем id после обработки
                    processed_ids.remove(obj_id)
        else:
            # Базовый случай: неитерируемый объект или строка
            yield nested_obj
    
    # Создаём генератор
    generator = _flatten_generator(nested_obj)
    
    # Возвращаем либо список, либо генератор в зависимости от параметра gen
    if gen:
        return generator
    else:
        return list(generator)

In [54]:
try:
    # Создаём циклический список
    cyclic_list = [1, 2, 3]
    cyclic_list.append(cyclic_list)  # Список ссылается сам на себя
    print(f"Создан циклический список: {cyclic_list}")
    
    result = flatten_it(cyclic_list)
    print(f"Результат: {result}")
except ValueError as e:
    print(f"Перехвачено исключение ValueError: {e}")
print()

Создан циклический список: [1, 2, 3, [...]]
Перехвачено исключение ValueError: Обнаружен циклический список: объект уже был обработан. Путь: ['list[3]']

